# Every number in the paper, recomputed

One place for all of the paper's numerical claims -- counts, percentages,
confidence intervals and significance tests -- each computed from the released
data and checked against the value the draft quotes.  The figure notebooks make
the figures; this one makes the numbers in the text, so none of them depend on
a calculation done once by hand.

Parsing and scoring reuse exactly what the figure notebooks use
(`models/utils/parse_lmm_output_utils.py`; the plot-type scoring of
`counting_types_figure.ipynb`; the log-ratio of `mean_med_calcs_figure*.ipynb`),
so a number here and the matching figure cannot disagree.

Each claim goes through `check(section, claim, computed, paper)`.  `paper` is
the value **as written in the draft** (a string, so its precision is explicit):

* `'10.4'` -- passes if `computed` rounds to it at that precision
* `'<0.001'` -- passes if `computed` is below the bound
* `'~80'` -- passes if within 5 of it (for the draft's "approximately")
* anything else -- exact string match

The last cell prints every check, and lists any that fail.  When the data or
the draft change, re-run top to bottom and read that table.

Sections: 1 dataset · 2 evaluation sample and its uncertainty · 3 answer
parsing (App. A) · 4 plot type and provenance (Fig. 2) · 5 angular height
(Fig. 3) · 6 published figures (Fig. 4) · 7 classifier baseline · 8 aging
claim · 9 example-question table (App. B) · 10 aging preset (App. C) ·
11 summary.

In [1]:
# ======================== CONFIG ========================
data_root    = '~/astro_sky_image_vqa/'
qa_jsons_dir = data_root + 'VQA_full/qa_jsons/'
realsky_dir  = data_root + 'VQA_realskyimages/'
runs = {
    'original':      data_root + 'LMM_outputs_n255/',
    'archive-light': data_root + 'LMM_outputs_n255_archive_light/',
}
published_run = data_root + 'LMM_outputs_realskyimages/'   # Gemini only

model_names = {'chatgpt_api': 'GPT', 'gemini': 'Gemini', 'claude_haiku': 'Claude'}

# feature table written by misc/provenance_classifier_baseline.ipynb
classifier_features_csv = '../../misc/provenance_features.csv'
# the generator, for the field-size policy defaults quoted in Sec. 2.1
generator_script = '../../create_figures_triplets_batch.py'

In [2]:
import os, re, json, glob, pickle, math, collections
import numpy as np
import pandas as pd

import sys as _sys
_sys.path.insert(0, os.path.abspath('..'))
from utils.parse_lmm_output_utils import (parse_llm_dual_json, answer_for,
                                          expected_key_from_format, ground_truth,
                                          as_float, PLACEHOLDER)

ex = os.path.expanduser
Z95 = 1.959964   # two-sided 95%


# ---------------------------------------------------------------- statistics
def wilson_halfwidth(n, p=0.5, z=Z95):
    """Half-width of the Wilson score interval, in percentage points.
    At p = 0.5 this is the widest the interval gets for a given n."""
    denom = 1 + z**2 / n
    return 100 * z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom


def two_prop_z(p1, n1, p2, n2):
    """Pooled two-proportion z-test, two-sided.  Returns (z, p)."""
    p = (p1 * n1 + p2 * n2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    return z, math.erfc(abs(z) / math.sqrt(2))


def min_detectable_diff(n, p=0.5, power_z=0.8416, alpha_z=Z95):
    """Smallest difference between two proportions (each from n) detectable at
    80% power and two-sided alpha = 0.05, in percentage points."""
    return 100 * (alpha_z + power_z) * math.sqrt(2 * p * (1 - p) / n)


# ---------------------------------------------------------------- checking
CHECKS = []

def check(section, claim, computed, paper):
    """Record one claim: the value computed here vs. the one in the draft."""
    paper = str(paper)
    if paper.startswith('<'):
        ok = computed < float(paper[1:])
    elif paper.startswith('~'):
        ok = abs(computed - float(paper[1:])) <= 5
    else:
        try:
            target = float(paper)
            nd = len(paper.split('.')[1]) if '.' in paper else 0
            ok = abs(float(computed) - target) <= 0.5 * 10**-nd + 1e-9
        except ValueError:
            ok = str(computed) == paper
    shown = ('%.4g' % computed) if isinstance(computed, (float, np.floating)) else computed
    CHECKS.append({'section': section, 'claim': claim, 'computed': shown,
                   'paper': paper, 'ok': ok})
    print('%-4s %-62s computed %-12s paper %s' % ('ok' if ok else 'FAIL', claim, shown, paper))

## 1. The dataset

Figure and question counts (abstract; Results ¶1), and the units remark in
Sec. 2.1.  Every released qa_json is read once here; the per-figure category
it yields is reused by every later section.

In [3]:
qa_files = sorted(glob.glob(os.path.join(ex(qa_jsons_dir), '*_qa.json')))

category = {}                 # vqa_id -> contour / sky-gmm / sky-real
n_questions = {}              # vqa_id -> questions in its qa_json
bunit = {}                    # vqa_id -> BUNIT string (real sky only)
records = {}                  # vqa_id -> parsed qa_json, kept for App. B

def count_questions(rec):
    n = 0
    for lvl in rec['VQA'].values():
        for items in lvl.values():
            for e in items.values():
                # figure-level entries hold 'Q' directly; plot-level ones nest
                # one level deeper, under 'plot0'
                n += 1 if 'Q' in e else sum(1 for v in e.values()
                                            if isinstance(v, dict) and 'Q' in v)
    return n

for fp in qa_files:
    rec = json.loads(json.load(open(fp)))
    vid, p = rec['vqa_id'], rec['plot0']
    category[vid] = ('contour' if p['type'] == 'contour'
                     else 'sky-gmm' if p.get('distribution') == 'gmm' else 'sky-real')
    n_questions[vid] = count_questions(rec)
    if category[vid] == 'sky-real':
        hdr = p['data']['data params'].get('sky image params', {}).get('header', '')
        m = re.search(r"BUNIT\s*=\s*'([^']*)'", hdr)
        bunit[vid] = m.group(1).strip() if m else None
    if vid in ('vqa_000001', 'vqa_000002', 'vqa_000004', 'vqa_000007'):
        records[vid] = rec

cats = pd.Series(category)
nq = pd.Series(n_questions)
print(cats.value_counts().to_string(), '\n')
print('questions per figure, by category:')
print(nq.groupby(cats).agg(['min', 'max', 'sum']).to_string(), '\n')

check('1 dataset', 'figures in the dataset', len(cats), '2001')
for c in ('contour', 'sky-gmm', 'sky-real'):
    check('1 dataset', 'figures: %s' % c, int((cats == c).sum()), '667')
check('1 dataset', 'sky figures (real + GMM)', int(cats.isin(['sky-gmm', 'sky-real']).sum()), '1334')
check('1 dataset', 'total questions', int(nq.sum()), '47357')
check('1 dataset', 'questions per contour figure', int(nq[cats == 'contour'].max()), '21')
check('1 dataset', 'questions per sky figure', int(nq[cats != 'contour'].max()), '25')
# The pre-revision draft said 33350: the sky questions alone (1334 x 25).
print('\nsky-only questions (the old, wrong total):', int(nq[cats != 'contour'].sum()))

have_unit = pd.Series(bunit).notna()
check('1 dataset', '% real-sky cutouts declaring BUNIT ("~20%")', 100 * have_unit.mean(), '~20')
print('   units declared:', pd.Series(bunit).dropna().str.upper().value_counts().head(6).to_dict())

sky-gmm     667
sky-real    667
contour     667 

questions per figure, by category:
          min  max    sum
contour    21   21  14007
sky-gmm    25   25  16675
sky-real   25   25  16675 

ok   figures in the dataset                                         computed 2001         paper 2001
ok   figures: contour                                               computed 667          paper 667
ok   figures: sky-gmm                                               computed 667          paper 667
ok   figures: sky-real                                              computed 667          paper 667
ok   sky figures (real + GMM)                                       computed 1334         paper 1334
ok   total questions                                                computed 47357        paper 47357
ok   questions per contour figure                                   computed 21           paper 21
ok   questions per sky figure                                       computed 25           paper 25

sky-on

In [4]:
# Field-size policy, Sec. 2.1: "from 50 times the smallest resolution up to
# 0.75 times the FOV".  Read from the generator's argparse defaults.
src = open(generator_script).read()
def _default(flag):
    m = re.search(r'"-%s".*?default=([0-9.]+)' % flag, src, flags=re.S)
    return float(m.group(1))
check('1 dataset', 'field-size floor (x smallest resolution)', _default('res_factor'), '50')
check('1 dataset', 'field-size ceiling (x survey FOV)', _default('fov_factor'), '0.75')

ok   field-size floor (x smallest resolution)                       computed 50           paper 50
ok   field-size ceiling (x survey FOV)                              computed 0.75         paper 0.75


## 2. The evaluation sample and its uncertainty

Results ¶1, Fig. 2 caption, Limitations.  The pickles in each run directory
name the evaluated figures; every model and run must cover the same 255.

All answers are loaded once here and reused by Secs. 3–5.

In [5]:
def load_run(run_dir, model_dir):
    """{vqa_id: [qa entries]} for one model in one run."""
    out = {}
    for fp in sorted(glob.glob(os.path.join(ex(run_dir), model_dir, '*_qa.pickle'))):
        payload = pickle.load(open(fp, 'rb'))
        out[os.path.basename(fp).removesuffix('_qa.pickle')] = (
            payload[0] if isinstance(payload, (list, tuple)) else payload)
    return out

answers = []                       # one row per (run, model, figure, question)
figure_sets = {}
for run, rdir in runs.items():
    for mdir, mname in model_names.items():
        per_fig = load_run(rdir, mdir)
        figure_sets[(run, mname)] = set(per_fig)
        for vid, qa in per_fig.items():
            for e in qa:
                raw = e.get('raw answer') or e.get('Response')
                answers.append({
                    'run': run, 'model': mname, 'vqa_id': vid,
                    'category': category[vid],
                    'question': (e.get('question') or '').lower(),
                    'context': (e.get('context') or '').lower(),
                    'truth': ground_truth(e),
                    'raw': raw,
                    'answer': answer_for(raw, expected_key_from_format(e.get('format'))),
                })
answers = pd.DataFrame(answers)

sample = next(iter(figure_sets.values()))
assert all(s == sample for s in figure_sets.values()), 'runs/models cover different figures'
sample_cats = cats[sorted(sample)]
per_model = answers.groupby(['run', 'model']).size()
print(per_model.to_string())

check('2 sample', 'figures evaluated', len(sample), '255')
for c in ('contour', 'sky-gmm', 'sky-real'):
    check('2 sample', 'evaluated figures: %s' % c, int((sample_cats == c).sum()), '85')
check('2 sample', 'sample as % of dataset ("about 13%")', 100 * len(sample) / len(cats), '13')
check('2 sample', 'questions per model per run', int(per_model.max()), '6035')
check('2 sample', 'answers over 3 models x 2 conditions', len(answers), '36210')

check('2 sample', 'Wilson 95% half-width, n=85, p=0.5 (pp)', wilson_halfwidth(85), '10.4')
check('2 sample', 'Wilson 95% half-width, n=170, p=0.5 (pp)', wilson_halfwidth(170), '7.4')
check('2 sample', 'min. detectable diff, n=85/cell, 80% power (pp)', min_detectable_diff(85), '21')

run            model 
archive-light  Claude    6035
               GPT       6035
               Gemini    6035
original       Claude    6035
               GPT       6035
               Gemini    6035
ok   figures evaluated                                              computed 255          paper 255
ok   evaluated figures: contour                                     computed 85           paper 85
ok   evaluated figures: sky-gmm                                     computed 85           paper 85
ok   evaluated figures: sky-real                                    computed 85           paper 85
ok   sample as % of dataset ("about 13%")                           computed 12.74        paper 13
ok   questions per model per run                                    computed 6035         paper 6035
ok   answers over 3 models x 2 conditions                           computed 36210        paper 36210
ok   Wilson 95% half-width, n=85, p=0.5 (pp)                        computed 10.4         paper 10.

## 3. Answer parsing (Results ¶2, Appendix A)

An answer is **unparsable** when `answer_for()` returns nothing.  Two kinds:
the response is not valid JSON at all, or it is valid JSON whose answer is an
explicit `null` (a refusal).  Both count as wrong in Fig. 2.

In [6]:
def failure_kind(raw):
    s, _ = parse_llm_dual_json(raw or '')
    if not s or s == PLACEHOLDER:
        return 'not JSON'
    try:
        d = json.loads(s)
    except (json.JSONDecodeError, TypeError):
        return 'not JSON'
    vals = [v for k, v in d.items() if 'explan' not in k.lower()]
    return 'null' if vals and all(v is None for v in vals) else 'other'

bad = answers[answers['answer'].isna()].copy()
bad['kind'] = bad['raw'].map(failure_kind)
table_a = bad.pivot_table(index='model', columns='run', values='vqa_id', aggfunc='size',
                          fill_value=0).reindex(list(model_names.values()))
display(table_a)
print(bad.groupby(['model', 'kind']).size().to_string())

check('3 parsing', 'unparsable answers, total', len(bad), '48')
check('3 parsing', 'unparsable, % of all answers', 100 * len(bad) / len(answers), '0.13')
check('3 parsing', 'unparsable: not valid JSON', int((bad.kind == 'not JSON').sum()), '21')
check('3 parsing', 'unparsable: explicit null', int((bad.kind == 'null').sum()), '27')
for m, (o, a) in {'GPT': (12, 22), 'Gemini': (4, 3), 'Claude': (5, 2)}.items():
    check('3 parsing', '%s unparsable, original' % m, int(table_a.loc[m, 'original']), str(o))
    check('3 parsing', '%s unparsable, archive-light' % m, int(table_a.loc[m, 'archive-light']), str(a))
check('3 parsing', "GPT's unparsable answers, both runs", int((bad.model == 'GPT').sum()), '34')
check('3 parsing', "...of which explicit null", int(((bad.model == 'GPT') & (bad.kind == 'null')).sum()), '27')
check('3 parsing', 'Gemini + Claude unparsable that are null (all are not JSON)',
      int(((bad.model != 'GPT') & (bad.kind == 'null')).sum()), '0')

run,archive-light,original
model,,
GPT,22,12
Gemini,3,4
Claude,2,5


model   kind    
Claude  not JSON     7
GPT     not JSON     7
        null        27
Gemini  not JSON     7
ok   unparsable answers, total                                      computed 48           paper 48
ok   unparsable, % of all answers                                   computed 0.1326       paper 0.13
ok   unparsable: not valid JSON                                     computed 21           paper 21
ok   unparsable: explicit null                                      computed 27           paper 27
ok   GPT unparsable, original                                       computed 12           paper 12
ok   GPT unparsable, archive-light                                  computed 22           paper 22
ok   Gemini unparsable, original                                    computed 4            paper 4
ok   Gemini unparsable, archive-light                               computed 3            paper 3
ok   Claude unparsable, original                                    computed 5            paper 5
o

## 4. Plot type and provenance (Fig. 2)

Scored exactly as `counting_types_figure.ipynb` does: the plot-type question
in its **list** variant (the one offering [contour, image of the sky]) for
every figure; the provenance question ([gaussian mixture model, real image of
the sky]) for sky figures.  Lower-case, spaces and underscores stripped, exact
match; an unparsable answer is wrong.

Columns: `contour` and `sky` from the plot-type question; `GMM` and `real`
from the provenance question.

In [7]:
def normalise(x):
    return str(x).lower().replace(' ', '').replace('_', '').strip()

is_ptype = (answers.question.str.contains('plot type')
            & answers.context.str.contains('please choose'))
is_prov = (answers.question.str.contains('underlying distribution')
           & ~answers.question.str.contains('color-axis')
           & answers.category.isin(['sky-gmm', 'sky-real']))

pt = answers[is_ptype].assign(column=lambda d: np.where(d.category == 'contour', 'contour', 'sky'))
pv = answers[is_prov].assign(column=lambda d: d.category.map({'sky-gmm': 'GMM', 'sky-real': 'real'}))
fig2 = pd.concat([pt, pv])
fig2['parsed'] = fig2['answer'].notna() & (fig2['answer'].astype(str).str.strip() != '')
fig2['correct'] = [normalise(t) == normalise(a) if p else False
                   for t, a, p in zip(fig2.truth, fig2.answer, fig2.parsed)]

acc = (fig2.groupby(['run', 'model', 'column'])['correct']
           .agg(['mean', 'size']).rename(columns={'mean': 'acc', 'size': 'n'}))
acc['acc'] *= 100
cols = ['contour', 'sky', 'GMM', 'real']
for run in runs:
    print('---', run, '---')
    print(acc.loc[run]['acc'].unstack()[cols].reindex(list(model_names.values())).round(1).to_string())
print('\nn per column:', acc.loc['original'].loc['GPT']['n'].to_dict())

A = lambda run, m, c: acc.loc[(run, m, c), 'acc']

--- original ---
column  contour    sky   GMM  real
model                             
GPT        65.9   52.9  12.9  89.4
Gemini     81.2  100.0  10.6  94.1
Claude     15.3  100.0  14.1  80.0
--- archive-light ---
column  contour   sky   GMM  real
model                            
GPT        56.5  71.8  11.8  94.1
Gemini     76.5  97.6   3.5  97.6
Claude     22.4  99.4   8.2  89.4

n per column: {'GMM': 85, 'contour': 85, 'real': 85, 'sky': 170}


In [8]:
# --- the sentences around Fig. 2 ---
S = '4 Fig. 2'
check(S, 'Claude contour, original', A('original', 'Claude', 'contour'), '15')
check(S, 'Claude contour, archive-light', A('archive-light', 'Claude', 'contour'), '22')
check(S, 'GPT contour, original', A('original', 'GPT', 'contour'), '66')
check(S, 'GPT contour, archive-light', A('archive-light', 'GPT', 'contour'), '56')
check(S, 'Gemini contour, original ("~80%")', A('original', 'Gemini', 'contour'), '~80')
check(S, 'Gemini contour, archive-light ("~80%")', A('archive-light', 'Gemini', 'contour'), '~80')
check(S, 'GPT sky plot type, original', A('original', 'GPT', 'sky'), '53')
check(S, 'Gemini/Claude sky plot type, lowest ("almost always")',
      min(A(r, m, 'sky') for r in runs for m in ('Gemini', 'Claude')), '~100')

gmm = [A(r, m, 'GMM') for r in runs for m in model_names.values()]
real = [A(r, m, 'real') for r in runs for m in model_names.values()]
check(S, 'GMM-sky identified correctly, range', '%.0f-%.0f' % (min(gmm), max(gmm)), '4-14')
check(S, 'real-sky identified correctly, range', '%.0f-%.0f' % (min(real), max(real)), '80-98')

# "the largest gaps do not [lie within the interval]": Claude vs Gemini on contours
z, p = two_prop_z(A('original', 'Gemini', 'contour') / 100, 85,
                  A('original', 'Claude', 'contour') / 100, 85)
check(S, 'Claude vs Gemini contour (original): z', z, '8.6')
check(S, 'Claude vs Gemini contour (original): p', p, '<1e-16')

# App. A: unparsable answers on these two questions -- Claude only
un = fig2[~fig2.parsed].groupby(['run', 'model']).size()
n_fig2 = fig2.groupby(['run', 'model']).size()
check(S, 'Fig. 2 answers per model per run', int(n_fig2.max()), '425')
check(S, 'Fig. 2 unparsable: Claude, original', int(un.get(('original', 'Claude'), 0)), '3')
check(S, 'Fig. 2 unparsable: Claude, archive-light', int(un.get(('archive-light', 'Claude'), 0)), '1')
check(S, 'Fig. 2 unparsable: GPT + Gemini, both runs',
      int(un[un.index.get_level_values('model') != 'Claude'].sum()), '0')

ok   Claude contour, original                                       computed 15.29        paper 15
ok   Claude contour, archive-light                                  computed 22.35        paper 22
ok   GPT contour, original                                          computed 65.88        paper 66
ok   GPT contour, archive-light                                     computed 56.47        paper 56
ok   Gemini contour, original ("~80%")                              computed 81.18        paper ~80
ok   Gemini contour, archive-light ("~80%")                         computed 76.47        paper ~80
ok   GPT sky plot type, original                                    computed 52.94        paper 53
ok   Gemini/Claude sky plot type, lowest ("almost always")          computed 97.65        paper ~100
ok   GMM-sky identified correctly, range                            computed 4-14         paper 4-14
ok   real-sky identified correctly, range                           computed 80-98        paper 80-98
o

## 5. Angular height (Fig. 3)

`R = log10(LMM / GT)`, defined only where both are positive.  Percentages are
of the answers that could be plotted (parsable and positive), as in
`mean_med_calcs_figure.ipynb`.

In [9]:
ah = answers[answers.question.str.contains('angular height')].copy()
ah['GT'] = ah['truth'].map(as_float)
ah['LMM'] = ah['answer'].map(as_float)
ok = ah.GT.gt(0) & ah.LMM.gt(0)
ah['R'] = np.where(ok, np.log10(ah.LMM.where(ok) / ah.GT.where(ok)), np.nan)
ah['plotted'] = ah['R'].notna()

nonpos = ah[ah.LMM.notna() & ~ah.LMM.gt(0)].groupby(['run', 'model']).size()
print('non-positive answers (cannot be logged):', nonpos.to_dict())
check('5 Fig. 3', 'non-positive answers: only GPT archive-light, one',
      str(nonpos.to_dict()), str({('archive-light', 'GPT'): 1}))

def pct(sel, cond):
    s = ah[sel & ah.plotted]
    return 100 * cond(s).mean(), len(s)

over1 = lambda s: s.R > 1
within2 = lambda s: s.R.abs() < np.log10(2)
for m, (o, a) in {'GPT': ('53', '60'), 'Claude': ('32', '32')}.items():
    for run, v in (('original', o), ('archive-light', a)):
        check('5 Fig. 3', '%s answers > +1 dex, %s' % (m, run),
              pct((ah.model == m) & (ah.run == run), over1)[0], v)

non-positive answers (cannot be logged): {('archive-light', 'GPT'): 1}
ok   non-positive answers: only GPT archive-light, one              computed {('archive-light', 'GPT'): 1} paper {('archive-light', 'GPT'): 1}
ok   GPT answers > +1 dex, original                                 computed 52.98        paper 53
ok   GPT answers > +1 dex, archive-light                            computed 60           paper 60
ok   Claude answers > +1 dex, original                              computed 32.35        paper 32
ok   Claude answers > +1 dex, archive-light                         computed 32.35        paper 32


In [10]:
# Split by source (real vs GMM sky), original figures
O = ah.run == 'original'
g_real, _ = pct(O & (ah.model == 'Gemini') & (ah.category == 'sky-real'), within2)
g_gmm, _ = pct(O & (ah.model == 'Gemini') & (ah.category == 'sky-gmm'), within2)
check('5 Fig. 3', 'Gemini within 2x, real sky', g_real, '95')
check('5 Fig. 3', 'Gemini within 2x, GMM sky', g_gmm, '94')

for m, (vr, vg, vp) in {'GPT': ('59', '47', '0.12'), 'Claude': ('41', '24', '0.01')}.items():
    pr, nr = pct(O & (ah.model == m) & (ah.category == 'sky-real'), over1)
    pg, ng = pct(O & (ah.model == m) & (ah.category == 'sky-gmm'), over1)
    z, p = two_prop_z(pr / 100, nr, pg / 100, ng)
    print('   %s: real %.1f%% (n=%d) vs GMM %.1f%% (n=%d), z=%.2f' % (m, pr, nr, pg, ng, z))
    check('5 Fig. 3', '%s > +1 dex, real sky' % m, pr, vr)
    check('5 Fig. 3', '%s > +1 dex, GMM sky' % m, pg, vg)
    check('5 Fig. 3', '%s real vs GMM: p' % m, p, vp)

ok   Gemini within 2x, real sky                                     computed 95.29        paper 95
ok   Gemini within 2x, GMM sky                                      computed 94.12        paper 94
   GPT: real 58.8% (n=85) vs GMM 47.0% (n=83), z=1.54
ok   GPT > +1 dex, real sky                                         computed 58.82        paper 59
ok   GPT > +1 dex, GMM sky                                          computed 46.99        paper 47
ok   GPT real vs GMM: p                                             computed 0.1244       paper 0.12
   Claude: real 41.2% (n=85) vs GMM 23.5% (n=85), z=2.46
ok   Claude > +1 dex, real sky                                      computed 41.18        paper 41
ok   Claude > +1 dex, GMM sky                                       computed 23.53        paper 24
ok   Claude real vs GMM: p                                          computed 0.01393      paper 0.01


## 6. Published figures (Fig. 4)

Selection counts from `VQA_realskyimages/selection_manifest.json`; Gemini's
answers on the 104 kept figures scored against the human-verified plot-box
ground truth, against its answers on our real-sky figures under
`archive-light` -- as in `mean_med_calcs_figure_realsky.ipynb`.

In [11]:
man = json.load(open(ex(realsky_dir + 'selection_manifest.json')))
c = man['counts']
check('6 Fig. 4', 'figures scraped (astrometry flag)', len(man['figures']), '163')
check('6 Fig. 4', 'removed by the automated astrometry check', c['excluded_astrometry_check'], '7')
check('6 Fig. 4', 'left after that check', len(man['figures']) - c['excluded_astrometry_check'], '156')
check('6 Fig. 4', 'discarded at plot-box review', c['excluded_plot_box_review'], '52')
check('6 Fig. 4', 'human-verified figures kept', c['included'], '104')

# ground truth for the published figures, from their qa_jsons
pub_truth = {}
for fp in glob.glob(os.path.join(ex(realsky_dir), 'qa_jsons', '*_qa.json')):
    rec = json.loads(json.load(open(fp)))
    pub_truth[rec['vqa_id']] = {n: as_float(e['A'][n]) for lvl in rec['VQA'].values()
                                for qs in lvl.values() for n, e in qs.items()}

rows = []
for vid, qa in load_run(published_run, 'gemini').items():
    if vid not in pub_truth:
        continue                      # not human-verified
    for e in qa:
        q = (e.get('question') or '').lower()
        for lab, key in (('width', 'field width'), ('height', 'field height')):
            if 'angular ' + lab in q:
                rows.append({'source': 'published', 'q': lab, 'GT': pub_truth[vid][key],
                             'LMM': as_float(answer_for(e.get('raw answer') or e.get('Response'),
                                                        expected_key_from_format(e.get('format'))))})
syn = answers[(answers.run == 'archive-light') & (answers.model == 'Gemini')
              & (answers.category == 'sky-real')]
for lab in ('width', 'height'):
    for _, r in syn[syn.question.str.contains('angular ' + lab)].iterrows():
        rows.append({'source': 'ours (archive-light)', 'q': lab,
                     'GT': as_float(r.truth), 'LMM': as_float(r.answer)})
pub = pd.DataFrame(rows)
ok = pub.GT.gt(0) & pub.LMM.gt(0)
pub['R'] = np.where(ok, np.log10(pub.LMM.where(ok) / pub.GT.where(ok)), np.nan)

summ = (pub.dropna(subset=['R']).groupby(['q', 'source'])['R']
           .agg(n='size', median='median',
                within2=lambda r: 100 * (r.abs() < np.log10(2)).mean()))
display(summ.round(3))
G = lambda q, s, k: summ.loc[(q, s), k]
P, Ours = 'published', 'ours (archive-light)'

check('6 Fig. 4', 'width: published n (one unparsable)', int(G('width', P, 'n')), '103')
check('6 Fig. 4', 'width: our real-sky n', int(G('width', Ours, 'n')), '85')
check('6 Fig. 4', 'width median R, published (dex)', G('width', P, 'median'), '-0.08')
check('6 Fig. 4', 'width median R, ours (dex)', G('width', Ours, 'median'), '-0.05')
check('6 Fig. 4', 'width within 2x, published', G('width', P, 'within2'), '55')
check('6 Fig. 4', 'width within 2x, ours', G('width', Ours, 'within2'), '51')
check('6 Fig. 4', 'height median R, published (dex)', G('height', P, 'median'), '-0.04')
check('6 Fig. 4', 'height median R, ours (dex)', G('height', Ours, 'median'), '-0.05')
check('6 Fig. 4', 'height within 2x, published', G('height', P, 'within2'), '76')
check('6 Fig. 4', 'height within 2x, ours', G('height', Ours, 'within2'), '94')
z, p = two_prop_z(G('height', Ours, 'within2') / 100, int(G('height', Ours, 'n')),
                  G('height', P, 'within2') / 100, int(G('height', P, 'n')))
print('height within 2x, ours vs published: z = %.2f' % z)
check('6 Fig. 4', 'height within 2x, ours vs published: p', p, '<0.001')

ok   figures scraped (astrometry flag)                              computed 163          paper 163
ok   removed by the automated astrometry check                      computed 7            paper 7
ok   left after that check                                          computed 156          paper 156
ok   discarded at plot-box review                                   computed 52           paper 52
ok   human-verified figures kept                                    computed 104          paper 104


n  median  within2
q      source                                    
height ours (archive-light)   85  -0.045   94.118
       published             104  -0.038   75.962
width  ours (archive-light)   85  -0.047   50.588
       published             103  -0.082   55.340

ok   width: published n (one unparsable)                            computed 103          paper 103
ok   width: our real-sky n                                          computed 85           paper 85
ok   width median R, published (dex)                                computed -0.08155     paper -0.08
ok   width median R, ours (dex)                                     computed -0.04743     paper -0.05
ok   width within 2x, published                                     computed 55.34        paper 55
ok   width within 2x, ours                                          computed 50.59        paper 51
ok   height median R, published (dex)                               computed -0.0382      paper -0.04
ok   height median R, ours (dex)                                    computed -0.04518     paper -0.05
ok   height within 2x, published                                    computed 75.96        paper 76
ok   height within 2x, ours                                         computed 94.12        paper 

## 7. Classifier baseline (Plot type and Provenance)

Reads the feature table written by `misc/provenance_classifier_baseline.ipynb`
(run that first if the CSV is missing) and repeats its CV for the two numbers
the paper quotes: all features but colour span, and concentration alone.  Same
model, folds and seed as that notebook, so the numbers match its table.

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

feat = pd.read_csv(classifier_features_csv)
y = (feat['label'] == 'real').astype(int).values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
def cv_acc(cols):
    pred = cross_val_predict(GradientBoostingClassifier(random_state=0),
                             feat[cols].values, y, cv=cv)
    return 100 * np.mean(pred == y)

check('7 classifier', 'sky figures classified', len(feat), '1334')
check('7 classifier', 'accuracy, field size + concentration + colour-min sign',
      cv_acc(['field size', 'concentration', 'colour min sign']), '85')
check('7 classifier', 'accuracy, concentration alone', cv_acc(['concentration']), '80')

ok   sky figures classified                                         computed 1334         paper 1334


ok   accuracy, field size + concentration + colour-min sign         computed 85.01        paper 85
ok   accuracy, concentration alone                                  computed 79.69        paper 80


## 8. The aging claim (Conclusions)

All 12 cells of Fig. 2 (3 models × 4 columns), original vs `archive-light`,
two-proportion tests, Bonferroni-corrected over the 12.  The draft says one
cell changes: GPT's sky-image plot type.

In [13]:
tests = []
for m in model_names.values():
    for c in cols:
        a0, n0 = acc.loc[('original', m, c), ['acc', 'n']]
        a1, n1 = acc.loc[('archive-light', m, c), ['acc', 'n']]
        z, p = two_prop_z(a0 / 100, int(n0), a1 / 100, int(n1))
        tests.append({'model': m, 'column': c, 'original': a0, 'archive-light': a1,
                      'n': int(n0), 'z': z, 'p': p, 'survives Bonferroni': p < 0.05 / 12})
tests = pd.DataFrame(tests)
display(tests.round(4))

survivors = tests[tests['survives Bonferroni']]
check('8 aging', 'cells that change after Bonferroni (of 12)', len(survivors), '1')
check('8 aging', '...which cell', ' '.join(survivors[['model', 'column']].iloc[0]), 'GPT sky')
g = tests[(tests.model == 'GPT') & (tests.column == 'sky')].iloc[0]
check('8 aging', 'GPT sky plot type, original', g['original'], '53')
check('8 aging', 'GPT sky plot type, archive-light', g['archive-light'], '72')
check('8 aging', 'GPT sky plot type change: p', g['p'], '<0.001')

,model,column,original,archive-light,n,z,p,survives Bonferroni
0,GPT,contour,65.8824,56.4706,85,1.2590,0.2080,False
1,GPT,sky,52.9412,71.7647,170,-3.5819,0.0003,True
2,GPT,GMM,12.9412,11.7647,85,0.2331,0.8157,False
3,GPT,real,89.4118,94.1176,85,-1.1160,0.2644,False
4,Gemini,contour,81.1765,76.4706,85,0.7509,0.4527,False
5,Gemini,sky,100.0000,97.6471,170,2.0119,0.0442,False
6,Gemini,GMM,10.5882,3.5294,85,1.7966,0.0724,False
7,Gemini,real,94.1176,97.6471,85,-1.1580,0.2469,False
8,Claude,contour,15.2941,22.3529,85,-1.1772,0.2391,False
9,Claude,sky,100.0000,99.4118,170,1.0015,0.3166,False


ok   cells that change after Bonferroni (of 12)                     computed 1            paper 1
ok   ...which cell                                                  computed GPT sky      paper GPT sky
ok   GPT sky plot type, original                                    computed 52.94        paper 53
ok   GPT sky plot type, archive-light                               computed 71.76        paper 72
ok   GPT sky plot type change: p                                    computed 0.0003411    paper <0.001


## 9. Example-question table (Appendix B)

Each example answer, read from the qa_json of the figure the table caption
names for it.

In [14]:
def answer_of(rec, name):
    """Ground truth for question `name` in a qa_json record."""
    for lvl in rec['VQA'].values():
        for items in lvl.values():
            if name in items:
                e = items[name]
                e = e if 'A' in e else e['plot0']
                a = e['A']
                # stored under the question name, or under the answer key the
                # prompt asked for (e.g. 'plot types' for 'plot types (list)')
                while isinstance(a, dict):
                    a = a[name] if name in a else next(iter(a.values()))
                return a[0] if isinstance(a, list) and len(a) == 1 else a
    raise KeyError(name)

pubrec = json.loads(json.load(open(ex(realsky_dir + 'qa_jsons/rsky_000001_qa.json'))))
r1, r2, r4, r7 = (records[v] for v in ('vqa_000001', 'vqa_000002', 'vqa_000004', 'vqa_000007'))
T = '9 App. B'
check(T, 'plot type (vqa_000001)', answer_of(r1, 'plot types (list)'), 'image of the sky')
check(T, 'colormap (vqa_000001)', answer_of(r1, 'colormap'), 'rainbow_r')
check(T, 'epoch (vqa_000004)', answer_of(r4, 'epoch'), 'J2000')
check(T, 'finest RA unit (vqa_000001)', answer_of(r1, 'finest unit right ascension'), 'minutes')
check(T, 'minimum RA axis limit (vqa_000001)',
      answer_of(r1, 'minimum right ascension axis limit'), '18h46m50.98s')
check(T, 'contour style (vqa_000007)', answer_of(r7, 'image or lines (list)'), 'contour lines')
check(T, 'angular height, arcmin (vqa_000002)', float(answer_of(r2, 'field height')), '1.58')
check(T, 'angular width, arcmin (vqa_000002)', float(answer_of(r2, 'field width')), '1.88')
check(T, 'mean colour (vqa_000001)', float(answer_of(r1, 'mean color')), '108.4')
check(T, 'provenance (vqa_000002)', answer_of(r2, 'distribution-image + list)'), 'real image of the sky')
check(T, 'contour colour distribution (vqa_000007)', answer_of(r7, 'distribution-color + list)'), 'linear')
check(T, 'pixel scale, arcsec (vqa_000001, corrected GT)', float(answer_of(r1, 'pixel scale')), '188.0')
check(T, 'RA of centre, deg (rsky_000001)', float(answer_of(pubrec, 'center right ascension')), '91.95')

ok   plot type (vqa_000001)                                         computed image of the sky paper image of the sky
ok   colormap (vqa_000001)                                          computed rainbow_r    paper rainbow_r
ok   epoch (vqa_000004)                                             computed J2000        paper J2000
ok   finest RA unit (vqa_000001)                                    computed minutes      paper minutes
ok   minimum RA axis limit (vqa_000001)                             computed 18h46m50.98s paper 18h46m50.98s
ok   contour style (vqa_000007)                                     computed contour lines paper contour lines
ok   angular height, arcmin (vqa_000002)                            computed 1.575        paper 1.58
ok   angular width, arcmin (vqa_000002)                             computed 1.876        paper 1.88
ok   mean colour (vqa_000001)                                       computed 108.4        paper 108.4
ok   provenance (vqa_000002)                   

## 10. Aging preset (Appendix C)

The "fired" column of the preset table, and the grayscale fraction (the
text's "2/3 of the images"), from the manifest of the `archive-light` run.

In [15]:
manifest = json.load(open(ex(runs['archive-light'] + 'archive_manifest.json')))
fired = collections.Counter(e for r in manifest.values() for e in r['effects'])
nfig = len(manifest)
check('10 App. C', 'figures in the aging manifest', nfig, '255')
paper_fired = {'InkBleed-soft': 55, 'InkMottling-soft': 34, 'ColorPaper-soft': 80,
               'NoiseTexturize-soft': 55, 'BrightnessTexturize-soft': 49,
               'Stains-soft': 38, 'Folding': 25, 'ShadowCast-soft': 24,
               'Geometric-soft': 49, 'SubtleNoise': 48, 'Jpeg-soft': 59}
for eff, v in paper_fired.items():
    check('10 App. C', 'fired: %s' % eff, 100 * fired[eff] / nfig, str(v))
gray = np.mean([r['grayscale'] for r in manifest.values()])
probs = {r['gray_prob'] for r in manifest.values()}
check('10 App. C', 'grayscale probability set ("2/3")', probs.pop(), '0.67')
check('10 App. C', 'grayscale realised, % of figures', 100 * gray, '~67')

ok   figures in the aging manifest                                  computed 255          paper 255
ok   fired: InkBleed-soft                                           computed 54.51        paper 55
ok   fired: InkMottling-soft                                        computed 33.73        paper 34
ok   fired: ColorPaper-soft                                         computed 80           paper 80
ok   fired: NoiseTexturize-soft                                     computed 54.51        paper 55
ok   fired: BrightnessTexturize-soft                                computed 48.63        paper 49
ok   fired: Stains-soft                                             computed 37.65        paper 38
ok   fired: Folding                                                 computed 25.1         paper 25
ok   fired: ShadowCast-soft                                         computed 23.92        paper 24
ok   fired: Geometric-soft                                          computed 48.63        paper 49
ok   fire

## 11. Summary

In [16]:
summary = pd.DataFrame(CHECKS)
print('%d checks, %d failed' % (len(summary), (~summary.ok).sum()))
fails = summary[~summary.ok]
if len(fails):
    print('\nFAILED -- the draft and the data disagree here:')
    display(fails)
pd.set_option('display.max_rows', 500)
summary.assign(ok=summary.ok.map({True: 'ok', False: 'MISMATCH'}))

114 checks, 0 failed


,section,claim,computed,paper,ok
0,1 dataset,figures in the dataset,2001,2001,ok
1,1 dataset,figures: contour,667,667,ok
2,1 dataset,figures: sky-gmm,667,667,ok
3,1 dataset,figures: sky-real,667,667,ok
4,1 dataset,sky figures (real + GMM),1334,1334,ok
5,1 dataset,total questions,47357,47357,ok
6,1 dataset,questions per contour figure,21,21,ok
7,1 dataset,questions per sky figure,25,25,ok
8,1 dataset,"% real-sky cutouts declaring BUNIT (""~20%"")",19.79,~20,ok
9,1 dataset,field-size floor (x smallest resolution),50,50,ok
